In [1]:
import os
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from itertools import product
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm
from functions_utils import *
from functions_data import *
from functions_optimize import *
from functions_eval import *
import time

In [ ]:
# 실험 설정
S_VALUES = [100, 200]
SEEDS = [1, 2, 3]
LEVELS = ["medium", "high"]
BASE_PATH = "/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result"

# 총 실험 수 계산
total_experiments = len(S_VALUES) * len(SEEDS) * len(LEVELS)
print(f"총 {total_experiments}개의 실험을 시작합니다.")
print(f"S: {S_VALUES}")
print(f"Seeds: {SEEDS}")
print(f"Levels: {LEVELS}")
print("=" * 80)

# 데이터 로딩 (한 번만)
print("📂 데이터 로딩 중...")
generation_data, I, T = load_generation_data(date_filter="2022-07-18")
print(f"✅ 데이터 로딩 완료: I={I}, T={T}")

experiment_count = 0
start_time = time.time()

for S, SEED, LEVEL in product(S_VALUES, SEEDS, LEVELS):
    experiment_count += 1
    exp_start_time = time.time()
    
    print(f"\n[실험 {experiment_count}/{total_experiments}] S={S}, SEED={SEED}, LEVEL={LEVEL}")
    print("-" * 60)
    
    try:
        # 기존 결과 확인
        exists, message = check_batch_exists(I, S, SEED, LEVEL, base_dir=BASE_PATH)
        if exists:
            print(f"⏭️  이미 존재함: {message}")
            continue
        
        # 파라미터 로딩
        print("🔧 파라미터 생성 중...")
        R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
        P_DA, P_PN = load_price_data(P_RT)
        
        # 시나리오 데이터 저장
        print("💾 시나리오 데이터 저장 중...")
        save_scenario_data(R, P_DA, P_RT, P_PN, I, T, S, SEED, LEVEL, base_dir=BASE_PATH)
        
        # Holistic 최적화
        print("🚀 Holistic 최적화 중...")
        x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, obj_hol = optimize_hol(
            R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2
        )
        
        # 결과 저장
        print("💾 최적화 결과 저장 중...")
        save_dir = save_holistic_results(
            x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, 
            ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, 
            obj_hol, I, T, S, SEED, LEVEL, base_dir=BASE_PATH
        )
        
        exp_time = time.time() - exp_start_time
        elapsed_time = time.time() - start_time
        avg_time = elapsed_time / experiment_count
        remaining_experiments = total_experiments - experiment_count
        estimated_remaining = avg_time * remaining_experiments
        
        print(f"✅ 완료! 목적함수값: {obj_hol:.2f}")
        print(f"⏱️  소요시간: {exp_time:.1f}초")
        print(f"📊 진행률: {experiment_count}/{total_experiments} ({100*experiment_count/total_experiments:.1f}%)")
        print(f"⏳ 예상 남은 시간: {estimated_remaining/60:.1f}분")
        
    except Exception as e:
        print(f"❌ 오류 발생: {str(e)}")
        print(f"   S={S}, SEED={SEED}, LEVEL={LEVEL}")
        continue

total_time = time.time() - start_time
print("\n" + "=" * 80)
print(f"🎉 모든 실험 완료!")
print(f"⏱️  총 소요시간: {total_time/60:.1f}분")
print(f"📁 결과 저장 위치: {BASE_PATH}")

# 완료된 실험 요약
print("\n📋 완료된 실험 요약:")
for S in S_VALUES:
    available_batches = get_available_batches(I, S, base_dir=BASE_PATH)
    print(f"S={S}: {len(available_batches)}개 배치 완료")
    for seed, level in available_batches:
        print(f"  - seed_{seed}_level_{level}")

총 12개의 실험을 시작합니다.
S: [100, 200]
Seeds: [1, 2, 3]
Levels: ['medium', 'high']
📂 데이터 로딩 중...
✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
✅ 데이터 로딩 완료: I=5, T=24

[실험 1/12] S=100, SEED=1, LEVEL=medium
------------------------------------------------------------
🔧 파라미터 생성 중...
📊 데이터 Shape: I=5, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='medium', Random Seed=1, M1=645.00, M2=1844.00
💾 시나리오 데이터 저장 중...
🔄 시나리오 데이터 저장 중... (폴더: /Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result\i_5_s_100\seed_1_level_medium)
✅ R.csv 저장 완료
✅ P_DA.csv 저장 완료
✅ P_RT.csv 저장 완료
✅ P_PN.csv 저장 완료
🎉 모든 시나리오 데이터가 '/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result\i_5_s_100\seed_1_level_medium' 폴더에 저장되었습니다!
📁 총 4개 파일 생성
🚀 Holistic 최적화 중...
Set parameter Username
Set parameter LicenseID to value 2681721
Academic license - for non-commercial use only - expires 2026-06-24
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11

KeyboardInterrupt: 

Exception ignored in: 'gurobipy._core.logcallbackstub'
Traceback (most recent call last):
  File "c:\Users\jangseohyun\anaconda3\Lib\site-packages\ipykernel\iostream.py", line 655, in write
    def write(self, string: str) -> Optional[int]:  # type:ignore[override]
KeyboardInterrupt: 
